# 🍩 Glaze & Classify — Interactive Explorer

Translate and classify bakery products with **Cortex AI**, or ask the **Intelligence Agent** questions about classification accuracy.

This notebook complements the [Streamlit dashboard](https://docs.snowflake.com/en/user-guide/ui-snowsight/streamlit-about) which shows the metrics, charts, and comparison tables.

---

In [ ]:
USE SCHEMA SNOWFLAKE_EXAMPLE.GLAZE_AND_CLASSIFY;
USE WAREHOUSE SFE_GLAZE_AND_CLASSIFY_WH;

## Live Classify

Enter any product name — in **any language** — and run the cell to translate and classify it with Cortex AI.

Try: `チョコレート グレーズド リング`, `Croissant aux amandes`, `Dona Glaseada Original`

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import ai_complete, ai_translate, lit
import json

session = get_active_session()

# ── Change this value and re-run the cell ──
product_name = "チョコレート グレーズド リング"

# Step 1: Translate to English
translated = session.range(1).select(
    ai_translate(lit(product_name), lit(""), lit("en")).alias("translated")
).collect()[0]["TRANSLATED"]

print(f"Translated: {product_name}  →  {translated}")

# Step 2: Classify with structured JSON output
prompt = (
    "Classify this bakery product into exactly one category and subcategory. "
    "Categories: Glazed, Frosted, Filled, Cake, Specialty, Seasonal, "
    "Beverages, Merchandise.\n\n"
    f"Product: {translated}"
)

result = session.range(1).select(
    ai_complete(
        model="claude-haiku-4-5",
        prompt=prompt,
        response_format={
            "type": "json",
            "schema": {
                "type": "object",
                "properties": {
                    "category": {"type": "string"},
                    "subcategory": {"type": "string"},
                },
                "required": ["category", "subcategory"],
            },
        },
    ).alias("result")
).collect()[0]["RESULT"]

parsed = json.loads(result) if isinstance(result, str) else result
print(f"Category:    {parsed['category']}")
print(f"Subcategory: {parsed['subcategory']}")

### Raw SQL version

The same operation as pure SQL — useful for embedding in pipelines.

In [ ]:
SELECT
    'チョコレート グレーズド リング'                    AS original,
    AI_TRANSLATE('チョコレート グレーズド リング', '', 'en') AS translated,
    AI_COMPLETE(
        model => 'claude-haiku-4-5',
        prompt => CONCAT(
            'Classify this bakery product into exactly one category and subcategory. ',
            'Categories: Glazed, Frosted, Filled, Cake, Specialty, Seasonal, Beverages, Merchandise.\n\n',
            'Product: ', AI_TRANSLATE('チョコレート グレーズド リング', '', 'en')
        ),
        response_format => TYPE OBJECT(category VARCHAR, subcategory VARCHAR)
    ) AS classification;

---

## Ask the Agent

Ask the **Glaze & Classify Assistant** anything about products, accuracy, markets, or how to improve classification results.

Change the `question` variable and re-run the cell.

Try:
- *Which market has the lowest accuracy?*
- *What products did every approach get wrong?*
- *How does accuracy differ for image-only products?*
- *Show me low-confidence predictions from the robust pipeline*

In [ ]:
from snowflake.snowpark.context import get_active_session
from IPython.display import Markdown, display
import json

session = get_active_session()

AGENT = "SNOWFLAKE_EXAMPLE.GLAZE_AND_CLASSIFY.GLAZE_CLASSIFIER_AGENT"

# ── Change this question and re-run the cell ──
question = "Which market has the lowest accuracy?"

request = json.dumps({
    "messages": [{"role": "user", "content": [{"type": "text", "text": question}]}],
    "stream": False,
})

raw = session.sql(
    "SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN(?, ?) AS response",
    params=[AGENT, request],
).collect()[0]["RESPONSE"]

parsed = json.loads(raw) if isinstance(raw, str) else raw
parts = [c["text"] for c in parsed.get("content", []) if c.get("type") == "text"]

if parts:
    display(Markdown("\n\n".join(parts)))
else:
    print(json.dumps(parsed, indent=2, default=str))

### Raw SQL version

Call the agent directly from SQL — the response is a JSON VARIANT.

In [ ]:
SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
    'SNOWFLAKE_EXAMPLE.GLAZE_AND_CLASSIFY.GLAZE_CLASSIFIER_AGENT',
    $${
        "messages": [{
            "role": "user",
            "content": [{"type": "text", "text": "What is the overall accuracy of each classification approach?"}]
        }],
        "stream": false
    }$$
) AS agent_response;